In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

True

##### Chat Model

In [3]:
chat = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite-preview",
    temperature = 0,
    max_output_tokens = 200,
    seed = 0
)

#### chain of runnables

> allows modularity

> each component is a runnable

> each runnable can be invoked, batched, streamed, transformed and composed into a pipeline

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [5]:
chat_template = ChatPromptTemplate.from_messages([
    (
        'human', # converts input to HumanMessagePromptTemplate
        'Tell me about {topic}. \n' + CommaSeparatedListOutputParser().get_format_instructions()
    )
])

In [6]:
chain = chat_template | chat | CommaSeparatedListOutputParser() # pipe symbol ('|'), takes output of one component as input to the next

In [7]:
chain.invoke({'topic': 'extinct species'})

['Dodo',
 'Woolly Mammoth',
 'Passenger Pigeon',
 'Tasmanian Tiger',
 'Great Auk',
 "Steller's Sea Cow",
 'Quagga',
 'Saber-toothed Cat',
 'Moa',
 'Carolina Parakeet']

In [8]:
type(chain) # chain itself belongs to runnables class

langchain_core.runnables.base.RunnableSequence

#### Pipe Chains

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [10]:
RunnablePassthrough().invoke([{1: 2}, 1, 2, 3, ['hi']]) # RunnablePassthrough: passes inputs through without any modifications

[{1: 2}, 1, 2, 3, ['hi']]

##### chat template

In [11]:
chat_template_skills = ChatPromptTemplate.from_template('''
    Give a list of 5 most important intermediate or higher level skills required for {profession}.
    List the name of skills only.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
    Suggest how to master the {skills}.
    Answer in as few words as possible.
''')

# from_template automatically treats input as a human message

In [12]:
str_parser = StrOutputParser()
passthrough = RunnablePassthrough()

##### chain 1

> output of `StrOutputParser` serves as input to `RunnablePassthrough`

In [13]:
chain_skills = chat_template_skills | chat | str_parser | {'skills': passthrough} # StrOutputParser is needed since the chat model returns an AIMessage object, not a plain string

In [14]:
chain_skills.invoke({'profession': 'AI Engineer'})

{'skills': '1. Deep Learning Frameworks (PyTorch/TensorFlow)\n2. MLOps and Model Deployment\n3. Natural Language Processing (NLP) and LLM Architecture\n4. Distributed Computing and Scalable Data Pipelines\n5. Advanced Mathematical Optimization and Linear Algebra'}

##### chain 2

In [15]:
chain_strategy = chat_template_strategy | chat | str_parser

In [16]:
display(Markdown(chain_strategy.invoke({'skills': 'Langchain, Docker'})))

**LangChain:**
1. Build RAG pipelines.
2. Master LCEL (LangChain Expression Language).
3. Study agentic workflows.
4. Read official documentation/cookbooks.

**Docker:**
1. Learn `Dockerfile` syntax.
2. Master `docker-compose`.
3. Practice containerizing apps.
4. Understand networking and volumes.

##### compose both chains

In [17]:
chain_learn_skills = chain_skills | chain_strategy

In [18]:
response = chain_learn_skills.invoke({'profession': 'AI Engineer'})

In [19]:
display(Markdown(response))

1. **Frameworks:** Build custom layers/loss functions; replicate research papers.
2. **MLOps:** Master Docker, Kubernetes, MLflow, and CI/CD pipelines.
3. **NLP/LLMs:** Implement Transformers from scratch; fine-tune via PEFT/LoRA.
4. **Distributed:** Learn Spark, Ray, and Dask; optimize data ingestion/sharding.
5. **Math:** Study Matrix Calculus, Convex Optimization, and Eigen-decomposition.

#### visualize chain

In [20]:
chain_learn_skills.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
             *             
             *             
             *             
      +-------------+      
      | Passthrough |      
      +-------------+      
             *             
             *             
             *      